# Vélib' station availability

Runs on whichever data the app is using. On sample data the patterns are invented, so only quote findings from real data.

In [ ]:
import sys
sys.path.insert(0, '../api')
import matplotlib.pyplot as plt
from store import make_store

store = make_store()
print('Data:', store.mode)

## How often are stations empty through the day?

In [ ]:
df = store.query('SELECT local_hour, AVG(CASE WHEN bikes = 0 THEN 1 ELSE 0 END) AS empty_share FROM snapshots GROUP BY 1 ORDER BY 1')
ax = df.plot(x='local_hour', y='empty_share', legend=False, title='Share of stations with no bikes, by Paris hour')
ax.set_xlabel('Hour'); ax.set_ylabel('Share empty'); plt.show()

## Which stations are empty most often?

In [ ]:
df = store.query('''SELECT s.name, AVG(CASE WHEN r.bikes = 0 THEN 1 ELSE 0 END) AS empty_share
FROM snapshots AS r JOIN stations AS s USING (station_id) GROUP BY 1 ORDER BY 2 DESC LIMIT 10''')
ax = df.sort_values('empty_share').plot.barh(x='name', y='empty_share', legend=False, title='Stations most often empty')
ax.set_xlabel('Share of readings empty'); ax.set_ylabel(''); plt.show()

## Do weekends look different?

In [ ]:
df = store.query('SELECT local_hour, is_weekend, AVG(bikes) AS bikes FROM snapshots GROUP BY 1, 2 ORDER BY 1')
ax = df.pivot(index='local_hour', columns='is_weekend', values='bikes').rename(columns={0: 'Weekday', 1: 'Weekend'}).plot(title='Average bikes per station')
ax.set_xlabel('Hour'); ax.set_ylabel('Bikes'); ax.legend(title=''); plt.show()